# 创建driver进程
- 简单来说，任何执行了 Ray 初始化代码的 Python 进程，就会成为 Driver 进程

## 💻 通过 ray.init() 启动

- 这是最常见和推荐的方式。当你在一个 Python 脚本中调用 ray.init() 时，该脚本所在的进程就会被初始化为一个 Driver 进程。
- <font color='red'>启动本地集群：</font> 如果你直接运行 ray.init() 而不带任何参数，Ray 会在你的本地机器上启动一个单节点的 Ray 集群，并将当前进程作为 Driver 连接上去。这非常适合开发和调试。

In [ ]:
import ray
# 启动一个本地 Ray 实例，当前进程成为 Driver
ray.init()
# ... 你的分布式代码 ...

- <font color='red'>连接现有集群：</font> 你可以通过向 ray.init() 传递一个地址参数（例如 ray.init(address='auto') 或 ray.init(address='<head-node-ip>:6379')），让当前进程作为一个 Driver 连接到一个已经存在的、远程的 Ray 集群。

## 📝 通过 ray start 命令行启动

- 这是一种更底层的启动方式。你可以在命令行使用 ray start --head 来启动一个 Ray Head 节点。然后，你可以在这个 Head 节点上直接运行你的 Python 脚本。

- 在这种情况下，执行脚本的进程同样会成为一个 Driver 进程，并自动连接到本地的 Ray 集群。不过，这种方式不如 ray.init() 灵活，通常用于特定的部署场景。

- 总而言之，Driver 进程的身份是在你调用 ray.init() 时被赋予的。这个调用是创建 Driver 进程并将其与 Ray 运行时关联起来的关键步骤。

# Driver 进程

在 Ray 分布式计算框架中，Driver 进程扮演着整个应用程序的“大脑”和“指挥中心”的角色。它是用户代码的入口，负责编排和协调整个分布式任务的执行。
## 🧠 Driver 进程的核心角色与职责

Driver 进程是执行用户主程序（例如 Python 的 __main__ 模块）的进程。它的主要职责包括：

- <font color='red'>代码执行与定义：</font>  Driver 负责执行用户的顶层代码，并在此过程中定义需要分布式执行的远程函数（Task，使用 @ray.remote 装饰的函数）和有状态的服务（Actor，使用 @ray.remote 装饰的类）。
- <font color='red'>任务提交与编排：</font>  Driver 通过调用 .remote() 方法将 Task 和 Actor 的创建请求提交到 Ray 集群中。它负责构建整个计算任务的有向无环图（DAG），决定任务的依赖关系和执行流程。
- <font color='red'>结果获取：</font>  当需要获取分布式任务的执行结果时，Driver 会调用 ray.get() 方法。这个调用是阻塞的，它会等待远程任务完成并返回最终结果。
- <font color='red'>生命周期管理：</font>  Driver 负责管理其所提交任务和 Actor 的生命周期，包括任务的提交、取消以及对结果的获取。

## ⚙️ Driver 进程的运行机制

从进程特性来看，Driver 有几个关键特点：

- 特殊的 Worker： Driver 本质上是一个特殊的工作进程。它可以提交任务，但自身不会执行任何由它提交的任务。
- 灵活的部署位置： <font color='red'>Driver 进程可以运行在 Ray 集群的任何一个节点上。不过，在默认情况下，它通常在 Head 节点上启动。</font>
- 与集群的连接： 用户可以<font color='red'>通过 ray.init() 来启动一个 Driver。这个调用可以启动一个嵌入式的单节点 Ray 实例，也可以让 Driver 连接到一个已存在的 Ray 集群。</font>



## 🤝 Driver 与其他核心组件的交互

Driver 进程并非孤立运行，它与 Ray 集群的其他核心组件紧密协作：

- 与 GCS (Global Control Store) 交互：
Driver 会与 GCS 通信，<font color='red'>GCS 是 Ray 集群的全局元数据存储中心</font>。
   - <font color='red'>Driver 通过 GCS 来注册和查找集群中的函数、任务和对象的位置信息。</font>

- 与<font color='red'> Raylet (本地调度器) </font>交互：
当 Driver 调用 .remote() 提交一个任务时，它首先会将任务提交给所在节点的本地调度器（Raylet）。
   - <font color='red'>Raylet 再根据集群的资源状况，决定是在本地执行还是将任务转发给全局调度器，最终分配给某个 Worker 进程去执行。</font>

- 与 Worker 进程交互：
   - <font color='red'>Driver 不直接与 Worker 通信来执行任务，而是通过调度系统（Raylet 和全局调度器）进行任务分配。当 Driver 调用 ray.get() 获取结果时，它会根据 ObjectRef 从分布式对象存储（Object Store）中拉取数据，这个过程可能涉及从远程 Worker 节点传输数据。</font>

总而言之，Driver 进程是 Ray 应用程序的控制中心，它通过定义、提交和协调分布式任务，驱动着整个 Ray 集群的计算过程。